# 第 1 周实验解答 —— 职位发布网页摘要

## 练习目标（理念）

抓取职位发布网页内容，交给 **OpenAI Chat Completions**，按你的说明筛出与 AI 相关的岗位并整理成 Markdown 列表。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | 同目录 `scraper.fetch_website_contents` |
| system / user prompt | `system_prompt` + `user_prompt_prefix` |
| `messages` 组装 | `messages_for(website)` |
| 调用 API 并展示 | `summarize` → `display_summary` + `Markdown` |

## 怎么跑

1. 配置 `.env` 中的 `OPENAI_API_KEY`（以 `sk-` 开头、无首尾空格）
2. 把 `url` 改成真实职位页地址
3. 依次运行单元格，最后执行 `display_summary(url)`


In [1]:
# ========== 导入：密钥、抓取、展示、OpenAI SDK ==========

# 导入标准库 os：读环境变量（Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境
from dotenv import load_dotenv
# 从同目录 scraper 导入抓取函数：取网页正文供后续喂给模型
from scraper import fetch_website_contents
# 在笔记本里用 Markdown 漂亮展示模型输出
from IPython.display import Markdown, display
# OpenAI 官方 Python 客户端
from openai import OpenAI


In [ ]:
# ========== 加载并校验 OPENAI_API_KEY ==========

# override=True：.env 中的值覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境读取密钥；不要把真实 key 写进笔记本
api_key = os.getenv("OPENAI_API_KEY")

# 三道门禁：缺失 / 不像 sk- 密钥 / 首尾有空白 —— 文案保持英文（可能被程序/用户依赖）
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")
elif not api_key.startswith("sk-"):
    raise ValueError("OPENAI_API_KEY environment variable is not valid")
elif api_key.strip() != api_key:
    raise ValueError("OPENAI_API_KEY environment variable contains leading or trailing whitespace")
else:
    # 通过校验时打印提示（字符串保持原样）
    print("OPENAI_API_KEY environment variable is set correctly")


In [3]:
# ========== 创建 OpenAI 客户端 ==========

# 默认会读环境变量 OPENAI_API_KEY；前面单元格已校验过
openai = OpenAI()


In [ ]:
# ========== 目标网址：改成你要分析的职位页 ==========

# 占位字符串；运行前替换为真实 URL（影响抓取结果，保持可运行原文）
url = "<Your URL here>"


In [4]:
# ========== Prompt：system 定角色，user 前缀定任务 ==========

# system_prompt 保留英文：发给模型的指令，改译会改变行为
system_prompt = """
You are a professional assistant that analyzes the contents of a website,
and provides a professional, detailes summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# user 前缀保留英文：具体任务说明（筛 AI 岗、字段、签证、按经验排序等）
user_prompt_prefix = """
Here are the contents of a job posting website.
If the job is related to AI, than list all those job posting with company name, role, experience expecting,work mode, salary(converting it to INR), skills required, responsibilities etc.
Also check whether they are offering the visa and sort them based on experiece, Contenet : .

"""


In [5]:
# ========== 组装 messages：把网页正文接到 user 前缀后面 ==========

def messages_for(website):
    # 返回 Chat Completions 需要的 system + user 两条消息
    return [
        {"role": "system", "content": system_prompt},
        # website：抓取到的页面文本，拼在 user_prompt_prefix 后面
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [6]:
# ========== 总结流程：抓取 → 调 API → 取助手回复 ==========

def summarize(url):
    # 用 scraper 拉取该 URL 的网页内容（字符串）
    website = fetch_website_contents(url)
    # 调用云端 Chat Completions；model id 保持原文 gpt-4.1-mini
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    # 取出第一条 choice 里的助手文本
    return response.choices[0].message.content


In [7]:
# ========== 展示：把 summarize 的结果渲染成 Markdown ==========

def display_summary(url):
    # 先拿到完整摘要字符串
    summary = summarize(url)
    # 在笔记本输出区显示 Markdown
    display(Markdown(summary))


In [10]:
# ========== 入口：对上面的 url 跑一遍完整流水线 ==========

# 确保 url 已改成真实地址，且前面单元格都已运行
display_summary(url)


# Summary of AI/ML Job Postings from Just Join IT

The following AI/ML-related job postings have been extracted from the website "Just Join IT," focusing on details such as company name, role, experience expected, work mode, salary converted to INR, required skills, responsibilities, and visa offering status where available.

---

### 1. Python / AI Engineer  
**Company:** edrone  
**Location:** Kraków + 4 Locations  
**Experience Level:** Not explicitly mentioned (likely Mid-level)  
**Work Mode:** Not specified (likely hybrid or office-based)  
**Salary:** 17,000 - 22,000 PLN/month  
**Salary in INR:** ₹3,30,000 - ₹4,27,000 per month (1 PLN ≈ 19.4 INR)  
**Skills Required:**  
- Python  
- AWS  
- AI  
**Responsibilities:** Not detailed  
**Visa Support:** Not mentioned  
**Additional Info:** 17 days left to apply  

---

### 2. Technical Program Manager, Dropbox Dash  
**Company:** Dropbox  
**Location:** Poland (Remote)  
**Experience Level:** Manager / C-level implied  
**Work Mode:** Remote  
**Salary:** 16,433 - 22,233 PLN/month  
**Salary in INR:** ₹3,18,000 - ₹4,31,000 per month  
**Skills Required:**  
- Program Management  
- Documentation  
- Communication  
**Responsibilities:** Not detailed  
**Visa Support:** Not mentioned  
**Additional Info:** 19 days left to apply  

---

### 3. MLOps Engineer  
**Company:** TQLO SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ  
**Location:** Warszawa  
**Experience Level:** Not specified (most likely Mid-level)  
**Work Mode:** Not specified  
**Salary:** 130 - 150 PLN/hour  
**Salary in INR:** ₹2,522 - ₹2,910/hour  
**Skills Required:**  
- Machine Learning  
- Azure ML  
- Docker  
**Responsibilities:** Not detailed  
**Visa Support:** Not mentioned  
**Additional Info:** 19 days left to apply  

---

### 4. Expert AI Engineer  
**Company:** Lumicode Sp. z o.o. (Pentacomp Group)  
**Location:** Gdańsk  
**Experience Level:** Senior / Expert  
**Work Mode:** Not specified  
**Salary:** 150 - 170 PLN/hour  
**Salary in INR:** ₹2,910 - ₹3,300/hour  
**Skills Required:**  
- GenAI  
- AWS  
- Terraform  
**Responsibilities:** Not detailed  
**Visa Support:** Not mentioned  
**Additional Info:** 29 days left to apply  

---

### 5. Vertex AI / ML Engineer  
**Company:** Accenture  
**Location:** Kraków + 3 other locations  
**Experience Level:** Not mentioned  
**Work Mode:** Not specified  
**Salary:** Undisclosed  
**Skills Required:**  
- Machine Learning  
- Vertex AI  
- GCP (Google Cloud Platform)  
**Responsibilities:** Not detailed  
**Visa Support:** Not mentioned  
**Additional Info:** 4 days left to apply  

---

### 6. AI/ML Engineer  
**Company:** Infinite Services  
**Location:** Warszawa + 4 locations  
**Experience Level:** Not specified  
**Work Mode:** Not specified  
**Salary:** 160 - 200 PLN/hour  
**Salary in INR:** ₹3,104 - ₹3,880/hour  
**Skills Required:**  
- SQL  
- Azure  
- AI  
**Responsibilities:** Not detailed  
**Visa Support:** Not mentioned  
**Additional Info:** 25 days left to apply  

---

### 7. AI Engineer  
**Company:** SOFLAB TECHNOLOGY  
**Location:** Warszawa  
**Experience Level:** Not specified (likely Mid-level)  
**Work Mode:** Not specified  
**Salary:** 170 - 220 PLN/hour  
**Salary in INR:** ₹3,298 - ₹4,268/hour  
**Skills Required:**  
- Python  
- C#  
- Microservices (likely)  
**Responsibilities:** Not detailed  
**Visa Support:** Not mentioned  
**Additional Info:** New posting  

---

# Summary Notes:
- None of the job postings explicitly mention visa sponsorship/support.
- Salaries are primarily in PLN with conversion to INR provided (1 PLN ≈ 19.4 INR).
- Most roles expect knowledge of Python, cloud platforms (AWS, Azure, GCP), and AI/ML frameworks.
- Work mode is largely unspecified or includes remote options.
- Experience levels vary from Junior to Manager-level, with several postings lacking explicit experience requirements.
- Specific responsibilities are generally not detailed in the extracted content.
- The highest salary appears for "AI Engineer" at SOFLAB TECHNOLOGY (₹3,298 - ₹4,268/hour).
- The roles are primarily based in Poland with some remote work options.

---

In [ ]:
# （空单元格占位：可在此继续试验别的 URL 或改 prompt）
